# Fundamentos de Ingeniería de Datos
## Ingesta y Fundamentos del ciclo de vida de datos


```
Fuente de datos→ CSV → Google Drive → SQLite (RAW)
```



### Configuración del entorno del proyecto

Antes de empezar, montamos Google Drive e importamos `utils.py`, el módulo que centraliza las rutas y funciones que usaremos en **todos los notebooks del curso**. Ejecuta estas celdas al inicio de cada sesión.


In [ ]:
# ── Importar utils.py ──────────────────────────────────────────────────────

# utils.py contiene las rutas del proyecto y funciones compartidas.
# Vive en Utilidades/utils.py, en la raíz del repositorio.


import sys

import os
sys.path.append(os.path.abspath('../Utilidades'))   # el notebook se ejecuta desde Codigos/


from utils import (

    PROYECTO_BASE, PRIVADO_PATH, DATASETS_RAW_PATH, DB_PATH, inspeccionar_dataset, listar_tablas
)

print()
print('utils.py importado correctamente')
print(f'Proyecto base: {PROYECTO_BASE}')
print()


In [ ]:
# ── Importar librerías y crear carpetas base ──────────────────────────────

import os
import json
import sqlite3
import pandas as pd
from datetime import datetime


os.makedirs(DATASETS_RAW_PATH, exist_ok=True)


# Carpeta para los entregables/documentación de cada unidad

DOCS_PATH = f'{PROYECTO_BASE}/Documentacion'

os.makedirs(DOCS_PATH, exist_ok=True)


print()
print('Entorno listo')
print()
print(f'  Datasets raw : {DATASETS_RAW_PATH}')
print(f'  Documentación: {DOCS_PATH}')
print()

### PASO 1 — Configuración de la API de Kaggle

Para autenticarte con Kaggle necesitas tu **usuario** y tu **API key**. Puedes obtenerlos en [kaggle.com](https://www.kaggle.com) → Settings → API → **Create New Token**, lo que descarga un archivo `kaggle.json`.

Sube tu `kaggle.json` a Drive una sola vez y apunta la variable de entorno a esa ruta.


In [ ]:
# ── Opción recomendada: guardar kaggle.json en la carpeta Privado/ ──
# 1. Descarga tu token en kaggle.com → Settings → API → Create New Token
# 2. Cópialo a: Privado/kaggle.json  (esa carpeta está en .gitignore, nunca se sube)


import os

KAGGLE_JSON_PATH = f'{PRIVADO_PATH}/kaggle.json'


if os.path.exists(KAGGLE_JSON_PATH):
    os.environ['KAGGLE_CONFIG_DIR'] = PRIVADO_PATH   # carpeta que contiene kaggle.json
    print()
    print('kaggle.json encontrado en Privado/ — usando esa configuración.')
    print()

else:

    # ── Alternativa: ingreso interactivo  ──

    from getpass import getpass
    os.environ['KAGGLE_USERNAME'] = input('Usuario de Kaggle: ')
    os.environ['KAGGLE_KEY']      = getpass('API key de Kaggle (no se mostrará en pantalla): ')
    print()
    print('Credenciales cargadas solo en esta sesión (no se guardan en el notebook).')
    print()



In [ ]:
# Instalar la librería de Kaggle

!pip install kaggle --quiet
print()
print('----- Librería de Kaggle instalada -----')
print()


### PASO 2 — Explorar datasets disponibles en Kaggle

Antes de descargar, buscamos desde la API qué hay disponible sobre un tema de interés. Cambia `search='...'` por el campo que más te interese.


In [ ]:
# Nos aseguramos de tener una versión reciente de la librería kaggle:
# versiones antiguas y nuevas de la API no siempre exponen los mismos atributos
# en los objetos de dataset (por eso el código de abajo usa getattr con opciones).


!pip install -q --upgrade kaggle
print()
print('----- Librería kaggle actualizada -----')
print()


In [ ]:
import kaggle


TEMA_BUSQUEDA = 'videogame sales'   # cambia esto por tu campo de interés

print()
print(f'----- Buscando datasets sobre \"{TEMA_BUSQUEDA}\" en Kaggle -----\n')
print()

datasets = kaggle.api.dataset_list(search=TEMA_BUSQUEDA, sort_by='votes', file_type='csv')


for i, ds in enumerate(datasets[:10]):


    tamano = getattr(ds, 'size', None) or getattr(ds, 'totalBytes', None) or 'N/D'
    votos  = getattr(ds, 'voteCount', 'N/D')
    print(f'{i+1}. {ds.ref}')
    print(f'   {ds.title}  |  {tamano}  |  votos: {votos}')


### PASO 3 — Descarga de datasets

Vamos a descargar los datasets que usaremos en el curso. El primero (`raw_df1`) será el dataset **principal**, el que recorrerá las 4 capas del pipeline en los siguientes notebooks.

In [ ]:
import os

# Crear carpeta de datos crudos (raw)

os.makedirs(DATASETS_RAW_PATH, exist_ok=True)

print(f'----- Carpeta lista: {DATASETS_RAW_PATH} -----')


In [ ]:
# ── Datasets a descargar ────────────────────────────────────────────────────

# Coloca el identificador tal como aparece en Kaggle (usuario/nombre-dataset).
# El PRIMER elemento de la lista es tu dataset principal (df1): será el que
# uses en TODOS los notebooks siguientes del curso.

mis_datasets = {
    'vg_ventas_ratings':    'thedevastator/video-game-sales-and-ratings',
    'vg_global_codificado': 'migeruj/data-global-videogames-sales-and-ratings',
    'vg_engagement':        'bimalkumarsaini/video-game-sales-and-engagement-analysis',
}


In [ ]:
# ── Descargar cada dataset ──────────────────────────────────────────────────

for etiqueta, ref in mis_datasets.items():
    nombre  = ref.replace('/', '_')
    carpeta = os.path.join(DATASETS_RAW_PATH, nombre)
    os.makedirs(carpeta, exist_ok=True)


    print()
    print(f'Descargando: {ref}')
    !kaggle datasets download -d {ref} -p {carpeta} --unzip --quiet
    print(f'   Listo → {carpeta}\n')
    print()


In [ ]:
# ── Cargar automáticamente el primer CSV de cada dataset descargado ────────

import glob

dataframes = {}

for etiqueta, ref in mis_datasets.items():
    carpeta = os.path.join(DATASETS_RAW_PATH, ref.replace('/', '_'))
    csvs = sorted(glob.glob(os.path.join(carpeta, '*.csv')))

    if not csvs:
        print(f'  Sin archivos .csv en {carpeta}, revisa la descarga.')
        continue

    print(f'\n{etiqueta}: CSV encontrados → {[os.path.basename(c) for c in csvs]}')

    try:
        dataframes[etiqueta] = pd.read_csv(csvs[0])
    except UnicodeDecodeError:
        dataframes[etiqueta] = pd.read_csv(csvs[0], encoding='latin-1')

    print(f'{etiqueta} ← {os.path.basename(csvs[0])}  '
          f'({dataframes[etiqueta].shape[0]:,} filas, {dataframes[etiqueta].shape[1]} columnas)')

# DataFrames con el nombre de su etiqueta
vg_ventas_ratings    = dataframes.get('vg_ventas_ratings')
vg_global_codificado = dataframes.get('vg_global_codificado')
vg_engagement        = dataframes.get('vg_engagement')


### PASO 4 — Carga e inspección inicial de los datos (Preparación)

Con los datos ya en memoria, hacemos la **inspección inicial**: dimensiones, tipos de dato y nulos. Esto es la etapa de *preparación* del ciclo de vida — todavía no modificamos nada, solo **conocemos** el dataset. Usamos `inspeccionar_dataset()` de `utils.py`.


In [ ]:
for etiqueta, df in dataframes.items():

    inspeccionar_dataset(df, nombre=f'{etiqueta} — {mis_datasets[etiqueta]}')


### PASO 5 — Almacenamiento en SQLite (capa RAW → Archivado)

Como buena práctica en ingeniería de datos, los datos ingeridos se guardan **sin modificaciones** en una capa raw. Esta es la etapa de *archivado*: los datos quedan persistidos de forma estable para que las siguientes capas del pipeline (EXTRACTED, TRANSFORMED, SERVING) puedan leer de aquí, siempre respetando la regla de oro:

> **Cada capa lee de la capa inmediatamente anterior. La capa RAW nunca se modifica.**


In [ ]:
import sqlite3

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
conn = sqlite3.connect(DB_PATH)

# Borrar las tablas con los nombres anteriores
for vieja in ['raw_video_games_df', 'raw_datasetnumeral_df', 'raw_games_df']:
    conn.execute(f'DROP TABLE IF EXISTS "{vieja}"')

for etiqueta, df in dataframes.items():
    tabla = f'raw_{etiqueta}'
    df.to_sql(tabla, conn, if_exists='replace', index=False)
    print(f'{etiqueta} → tabla "{tabla}" ({df.shape[0]:,} filas)')

conn.commit()
conn.close()
print(f'\nBase de datos: {DB_PATH}')

In [ ]:
# ── Verificar las tablas creadas en la base de datos ───────────────────────
listar_tablas(DB_PATH)


### PASO 6 — Consumo: primera consulta sobre los datos ya archivados

Cerramos el ciclo de vida con un primer **consumo**: una consulta SQL directa sobre la capa RAW, tal como lo haría cualquier persona que ya tiene acceso a los datos ingeridos.


In [ ]:
conn = sqlite3.connect(DB_PATH)


conn = sqlite3.connect(DB_PATH)

# Columna de género de cada tabla (cada dataset la llama distinto)
consultas = {
    'raw_vg_ventas_ratings':    'Genre',
    'raw_vg_global_codificado': 'Genre',
    'raw_vg_engagement':        'Genres',
}

for tabla, col in consultas.items():
    query = f'''
        SELECT "{col}" AS genero, COUNT(*) AS num_registros
        FROM "{tabla}"
        WHERE "{col}" IS NOT NULL
        GROUP BY "{col}"
        ORDER BY num_registros DESC
    '''
    resultado = pd.read_sql_query(query, conn)
    print(f'\nConsulta — conteo de registros por género en {tabla}:\n')
    display(resultado)

conn.close()

